# Stage 2: profile FinQA and ConvFinQA

Thin viewer over `python -m capstone.data.profile` (all logic and tests live in `src/capstone/data/`).
It re-runs the profile, shows the comparison with `data/profile_expected.yaml`, and spot-checks two
examples from CLAUDE.md. Run inside Docker (`make jupyter`).

In [1]:
import json
from pathlib import Path
import pandas as pd
from capstone.data import profile as P
from capstone.data.program import execute, parse

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
status = P.run(ROOT)
result = json.loads((ROOT / "data/interim/profile/profile.json").read_text())
print("exit status:", status)

OK: all 15 files match /workspace/data/checksums.sha256


  loaded finqa/dataset/train.json: 6,251 records
  loaded finqa/dataset/dev.json: 883 records
  loaded finqa/dataset/test.json: 1,147 records
  loaded finqa/dataset/private_test.json: 919 records


  loaded convfinqa/data/train.json: 3,037 records
  loaded convfinqa/data/dev.json: 421 records
  loaded convfinqa/data/test_private.json: 434 records


  wrote data/interim/profile/ (profile.json + 3 CSVs: 8,281 FinQA rows, 3,458 conversations, 12,594 turns)
  wrote docs/profile_report.md
OK: 54/54 fields match
exit status: 0


## Comparison with the Nov 1 numbers

In [2]:
cmp = pd.DataFrame(result["comparison"])
print(cmp.status.value_counts().to_string())
cmp[cmp.status != "match"] if (cmp.status != "match").any() else cmp.head(12)

status
match    54


,field,expected,ours,status
0,finqa_split_counts.train,6251,6251,match
1,finqa_split_counts.dev,883,883,match
2,finqa_split_counts.test,1147,1147,match
3,finqa_split_counts.private_test,919,919,match
4,finqa_labelled,8281,8281,match
5,finqa_companies,137,137,match
6,finqa_years,1999-2019,1999-2019,match
7,finqa_ops.divide,5901,5901,match
8,finqa_ops.subtract,3676,3676,match
9,finqa_ops.add,1952,1952,match


## Pipeline fields (not in the Nov 1 reference)

In [3]:
pd.json_normalize(result["pipeline"]).T.rename(columns={0: "value"})

,value
conv_turn_length_mismatches,0
conv_selection_values_not_in_ops,3
conv_trace_with_selection.base,3455
conv_trace_with_selection.all_found,2127
conv_trace_with_selection.exactly_one,1754
conv_pool_with_selection.pool,1304
conv_pool_with_selection.of,3458
finqa_reexec_failure_causes.percent_cells,30
finqa_reexec_failure_causes.duplicate_row_name,2


## FinQA programs that do not re-execute

In [4]:
pd.DataFrame(result["finqa_reexec_failures"])[["id", "program", "gold", "ours", "failure_cause"]]

,id,program,gold,ours,failure_cause
0,ALLE/2016/page_29.pdf-1,"table_average(2016, none)",0.25000,25.000000,percent_cells
1,ABMD/2009/page_85.pdf-1,"table_min(expected volatility, none), table_ma...",0.27400,27.400000,percent_cells
2,MA/2009/page_120.pdf-4,"table_average(expected volatility, none)",0.36833,36.833333,percent_cells
3,MAA/2018/page_88.pdf-2,"table_max(dividend yield, none)",0.03884,3.884000,percent_cells
4,AON/2014/page_45.pdf-4,"table_average(operating margin, none), subtrac...",0.00867,-19.923333,percent_cells
5,LMT/2015/page_52.pdf-1,"table_average(operating margins, none)",0.11100,11.100000,percent_cells
6,LMT/2013/page_46.pdf-3,"table_average(operating margins, none)",0.16500,16.500000,percent_cells
7,AAPL/2014/page_67.pdf-3,"table_max(effective tax rate, none)",0.26200,26.200000,percent_cells
8,AAPL/2007/page_48.pdf-1,"table_max(gross margin percentage, none)",0.34000,34.000000,percent_cells
9,ALLE/2016/page_29.pdf-2,"table_max(2015, none)",0.27000,27.000000,percent_cells


## Spot checks
1. **Aon 2009** (data plan worked example): growth should be 1.74%.
2. **FinQA train[0]** (known label noise): the question asks for "the interest expense in 2009", but
   the evidence sentence (`text_1`) says $3.8M is how much annual interest expense would *change* if
   LIBOR moved 100 bp. Also, the human `answer` field says **380** while the program and `exe_ans`
   give **3.8**, so `answer` and `exe_ans` can disagree; we use `exe_ans` (Nov 1 definition).

In [5]:
conv = json.loads((ROOT / "data/raw/convfinqa/data/train.json").read_text())
aon = next(e for e in conv + json.loads((ROOT / "data/raw/convfinqa/data/dev.json").read_text())
           if e["id"] == "Single_AON/2009/page_46.pdf-3")
for q, p, a in zip(aon["annotation"]["dialogue_break"], aon["annotation"]["turn_program"],
                   aon["annotation"]["exe_ans_list"]):
    print(f"{q!r:70.70}  {p:45}  gold={a}  ours={execute(p, aon['table'])}")

'what was the total of risk and insurance brokerage services segment r  6305                                           gold=6305.0  ours=6305.0
'and what was that in 2008?'                                            6197                                           gold=6197.0  ours=6197.0
'what was, then, the change over the year?'                             subtract(6305, 6197)                           gold=108.0  ours=108.0
'and how much does this change represent in relation to the 2008 total  subtract(6305, 6197), divide(#0, 6197)         gold=0.01743  ours=0.017427787639180247


In [6]:
t0 = json.loads((ROOT / "data/raw/finqa/dataset/train.json").read_text())[0]
print(t0["id"])
print("question:", t0["qa"]["question"])
print("program: ", t0["qa"]["program"], "->", execute(t0["qa"]["program"], t0["table"]))
print("answer:  ", t0["qa"]["answer"])

ADI/2009/page_49.pdf-1
question: what is the the interest expense in 2009?
program:  divide(100, 100), divide(3.8, #0) -> 3.8
answer:   380
